# Sík Reissner–Mindlin-lemez az LLFEM-ben

A 12. előadás 77–78. PDF-oldalának alakját használjuk. A középfelület az $xy$ síkban fekszik, $(\xi,\eta,\zeta)=(x,y,z)$. A mezők $U=(u_0,v_0)$, $W=w_0$, $\Phi=(\varphi_\xi,\varphi_\eta)$. A forgások a fizikai tengelyekhez tartoznak. A vastagság $b$.

$$
(u,v,w)=(u_0+\zeta\varphi_\eta,\ v_0-\zeta\varphi_\xi,\ w_0).
$$

Ebből a három alakváltozási blokk

$$
\varepsilon_0 =
\begin{bmatrix} u_{0,\xi}\\ v_{0,\eta}\\ u_{0,\eta}+v_{0,\xi} \end{bmatrix},
\qquad
\kappa =
\begin{bmatrix} \varphi_{\eta,\xi}\\ -\varphi_{\xi,\eta}\\
                 \varphi_{\eta,\eta}-\varphi_{\xi,\xi} \end{bmatrix},
\qquad
\gamma =
\begin{bmatrix} w_{0,\eta}-\varphi_\xi\\ w_{0,\xi}+\varphi_\eta \end{bmatrix}.
$$

A nyírás sorrendje $(\eta\zeta,\zeta\xi)$, ahogy az előadásban. A potenciális energia stacionaritásából a következő gyenge alak adódik

$$
\int_A \bigl[
 \delta\varepsilon_0^T D_1\varepsilon_0+
 \delta\kappa^T D_2\kappa+
 \delta\gamma^T D_3\gamma
 \bigr]\,dA
 = \int_A \delta d^T F_A\,dA + \int_{\Gamma_t}\delta d^T T_s\,ds,
 \qquad d=(u_0,v_0,w_0,\varphi_\xi,\varphi_\eta)^T.
$$

A tesztmezők homogén feltételt teljesítenek az előírt komponenseken. Szimmetrikus, homogén keresztmetszetnél $D_1=bC$, $D_2=b^3 C/12$, $D_3=k_s b G I_2$, ahol $C$ a síkfeszültségi mátrix és $G=E/[2(1+\nu)]$. Az előadás alakja $k_s=1$.

In [1]:
using LowLevelFEM
using LinearAlgebra

## Merevség

A **Grad** kétkomponensű mezőnél komponensenként adja a két deriváltat. Például
$\operatorname{Grad}\Phi=[\varphi_{\xi,\xi},\varphi_{\xi,\eta},\varphi_{\eta,\xi},\varphi_{\eta,\eta}]^T$.
Az alábbi mátrixok a három blokkot nyolc sorba rendezik. A $B$ mezősorrendje adja a terhelés és a megoldás sorrendjét is.

In [2]:
"""
    assemble_plate(mat; b, k_s=1.0)

Assemble the membrane, bending, and transverse-shear terms of an xy-plane
Reissner–Mindlin plate. The rotation field stores physical rotations about
the x and y axes. Use the material's physical surface group as the domain.
"""
function assemble_plate(mat::Material; b, k_s=1.0)
    body = mat.phName
    U = Field([mat]; type=:VectorField, dim=2, field=:u)
    W = Field([mat]; type=:ScalarField, dim=2, field=:w)
    Φ = Field([mat]; type=:VectorField, dim=2, field=:φ)

    # Rows 1:3: membrane strain, rows 4:6: curvature, rows 7:8: shear.
    A_u = vcat([1 0 0 0;
                0 0 0 1;
                0 1 1 0], zeros(5, 4))
    A_w = vcat(zeros(6, 2), [0 1; 1 0])
    A_φ = vcat(zeros(3, 4),
               [ 0  0  1  0;
                 0 -1  0  0;
                -1  0  0  1], zeros(2, 4))
    G_φ = vcat(zeros(6, 2), [-1 0; 0 1])

    B = A_u ⋅ Grad(U) + A_w ⋅ Grad(W) + A_φ ⋅ Grad(Φ) + G_φ ⋅ Φ

    C = D(:PlaneStress, mat)
    D1 = b * C
    D2 = (b^3 / 12) * C
    S = k_s * b * mat.μ
    D3 = S * Matrix{Float64}(I, 2, 2)
    D_plate = cat(D1, D2, D3; dims=(1, 2))
    K = ∫(B' ⋅ D_plate ⋅ B; Ω=body)

    return (; K, U, W, Φ, body, b, D1, D2, S)
end

assemble_plate (generic function with 1 method)

## Terhelés és megfogás

A felületi teher $F_A=(bf_\xi,bf_\eta,bf_\zeta+p^- -p^+,0,0)^T$.
A peremen $n=(n_\xi,n_\eta)$ a kifelé mutató **síkbeli élnormális**, és
$t=(-n_\eta,n_\xi)$. A dián szereplő $(N_n,N_{tn},Q_n,M_n,M_{tn})$
ebből a sorrendben $(N_n n+N_{tn}t,\ Q_n,\ M_n t-M_{tn}n)$ terhelést ad a három mezőre.
Az élnyomaték előjele a fizikai $\Phi$ forgásokhoz tartozik.

In [3]:
"""
    plate_area_load(plate; fξ=0, fη=0, fζ=0, p_minus=0, p_plus=0)

Return the three field load vectors for body forces and bottom-minus-top
pressure. The pressure may be supplied as a scalar field on the plate.
"""
function plate_area_load(plate; fξ=0.0, fη=0.0, fζ=0.0, p_minus=0.0, p_plus=0.0)
    (; U, W, Φ, body, b) = plate
    q = b*fζ + p_minus - p_plus
    return (
        ∫(U ⋅ [b*fξ, b*fη]; Ω=body),
        ∫(W ⋅ q; Ω=body),
        ∫(Φ ⋅ [0.0, 0.0]; Ω=body),
    )
end

"""
    plate_edge_load(plate, edge, n; Nn=0, Ntn=0, Qn=0, Mn=0, Mtn=0)

Return the three field load vectors for constant resultants on a straight
edge with outward unit normal n. Forces and moments use Cartesian components.
"""
function plate_edge_load(plate, edge, n; Nn=0.0, Ntn=0.0, Qn=0.0, Mn=0.0, Mtn=0.0)
    (; U, W, Φ) = plate
    t = [-n[2], n[1]]
    return (
        ∫(U ⋅ (Nn*n + Ntn*t); Γ=edge),
        ∫(W ⋅ Qn; Γ=edge),
        ∫(Φ ⋅ (Mn*t - Mtn*n); Γ=edge),
    )
end

plate_edge_load (generic function with 1 method)

## Példa szinuszos nyomásterheléssel

Egységnégyzet $b=0.1$ vastagsággal, Q9 hálón. A négy élen $w_0=0$ és
$\Phi\cdot n=0$, a másik forgáskomponenshez $M_n=0$ természetes
feltétel tartozik. A síkbeli mezőt bal oldalt rögzítjük a merevtestmódusok
kizárásához. $q=q_0\sin(\pi x)\sin(\pi y)$ esetén a pontos középponti
lehajlás $q_0/(D_{2,11}\lambda^2)+q_0/(S\lambda)$, ahol $\lambda=2\pi^2$
és $S=D_{3,11}$.

In [4]:
structured_rect_mesh(n=8, order=2)
mat = Material("body"; E=1.0, ν=0.3)
plate = assemble_plate(mat; b=0.1)
(; K, U, W, Φ, D2, S) = plate

q0 = D2[1, 1]
pressure = scalarField(W, "body", (x, y, z) -> q0*sinpi(x)*sinpi(y))
F = SystemVector([plate_area_load(plate; p_minus=pressure)...])

support = [BoundaryCondition(edge; field=W, w=0.0)
           for edge in ("left", "right", "bottom", "top")]
append!(support, [BoundaryCondition(edge; field=Φ, φx=0.0)
                  for edge in ("left", "right")])
append!(support, [BoundaryCondition(edge; field=Φ, φy=0.0)
                  for edge in ("bottom", "top")])
push!(support, BoundaryCondition("left"; field=U, ux=0.0, uy=0.0))

u, w, φ = solveField(K, F; support)
λ = 2π^2
w_exact = q0/(D2[1, 1]*λ^2) + q0/(S*λ)
w_center = probe(w, 0.5, 0.5, 0.0)
relative_error = abs(w_center / w_exact - 1)
@assert relative_error < 0.01
(; w_center, w_exact, relative_error)

(w_center = 0.00268541, w_exact = 0.002687116020388153, relative_error = 0.0006348889944494118)

## A peremnyomaték előjelének ellenőrzése

A következő próbán a középfelület síkbeli elmozdulása affin, a lehajlás
másodfokú, a forgás pedig $\varphi_\xi=w_{0,\eta}$,
$\varphi_\eta=-w_{0,\xi}$. Így a nyírás zérus, a membráneredők és
hajlítónyomatékok állandók. A próba nemzérus $N_{tn}$ és $M_{tn}$
mellett ellenőrzi az élerők és az élnyomatékok előjelét. Ugyanazt a hálót
és merevséget használjuk, mint az előző cellában.

In [5]:
εx, εy, γxy = 0.010, -0.005, 0.007
kxx, kxy, kyy = 0.12, 0.08, -0.04

ux(x, y, z) = εx*x + γxy*y/2
uy(x, y, z) = εy*y + γxy*x/2
w0(x, y, z) = (kxx*x^2 + 2kxy*x*y + kyy*y^2)/2
φx(x, y, z) = kxy*x + kyy*y
φy(x, y, z) = -kxx*x - kxy*y

Nξ, Nη, Nξη = plate.D1 * [εx, εy, γxy]
Mξ, Mη, Mξη = plate.D2 * [-kxx, -kyy, -2kxy]
N = [Nξ Nξη; Nξη Nη]
M = [Mξ Mξη; Mξη Mη]

F_patch = let loads = plate_area_load(plate)
    for (edge, n) in (("right", [1.0, 0.0]),
                      ("bottom", [0.0, -1.0]),
                      ("top", [0.0, 1.0]))
        t = [-n[2], n[1]]
        edge_loads = plate_edge_load(plate, edge, n;
            Nn=dot(n, N*n), Ntn=dot(t, N*n), Mn=dot(n, M*n), Mtn=dot(t, M*n))
        loads = map(+, loads, edge_loads)
    end
    SystemVector([loads...])
end
support_patch = [
    BoundaryCondition("left"; field=U, ux=ux, uy=uy),
    BoundaryCondition("left"; field=W, w=w0),
    BoundaryCondition("left"; field=Φ, φx=φx, φy=φy),
]
u_patch, w_patch, φ_patch = solveField(K, F_patch; support=support_patch)

u_exact = vectorField(U, "body", [ux, uy, 0.0])
w_exact_field = scalarField(W, "body", w0)
φ_exact = vectorField(Φ, "body", [φx, φy, 0.0])
errors = (u=norm(u_patch.a - u_exact.a)/norm(u_exact.a),
          w=norm(w_patch.a - w_exact_field.a)/norm(w_exact_field.a),
          φ=norm(φ_patch.a - φ_exact.a)/norm(φ_exact.a))
@assert maximum(errors) < 1e-9
errors

(u = 1.0375045074139866e-14, w = 3.385256129470603e-13, φ = 3.4470444592168053e-13)

A három energiatag itt teljes integrálással fut. A vékony határban a
nyírási locking ellen külön kell ellenőrizni a választott közelítést.